In [15]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split,cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from sklearn.compose import ColumnTransformer

In [9]:
data = load_breast_cancer()
X= pd.DataFrame(data.data,columns=data.feature_names)
y= data.target

numerical_features = X.select_dtypes(include=['number']).columns.tolist()
print(numerical_features)

['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry', 'worst fractal dimension']


In [10]:
categorical_features = X.select_dtypes(include=['object','category']).columns.tolist()
print(categorical_features)

[]


In [12]:
# Numercial Pipeline
numerical_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy='median')),
    ("scaler",StandardScaler())
])

# Categorical preprocessing
categorical_pipeline=Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# combine preprocessing
preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_features),
    ('cat',categorical_pipeline, categorical_features)
])

#complete ML PIPELINE
model = Pipeline([
    ('preprocessor',preprocessor),
    ('classifier', LogisticRegression(max_iter=2000))
])

In [13]:
X_train, X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [16]:
cv = StratifiedKFold(n_splits=5,shuffle=True, random_state=42)
scores = cross_val_score(model,X_train,y_train,cv=cv,scoring='accuracy')
model.fit(X_train,y_train)
y_pred = model.predict(X_test)
print("Test accuracy:", accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))

Test accuracy: 0.9824561403508771
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

